# 04 — Hybrid cluster XAI (train artefacts)

Builds explainability artefacts for the **cluster-routed hybrid** from [02_hybrid_ensemble.ipynb](../03_ml_layer_hybrid/02_hybrid_ensemble.ipynb), mirroring the structure of [`06b_xai_ensemble.ipynb`](../../06b_xai_ensemble.ipynb).

**Prereqs:** Run **02** so `03_ml_layer_hybrid/artifacts/hybrid_cluster_bundle.joblib` exists.

**Data:** Full feature table from **`hf_data/02_feature_layer/training/outputs/`** — latest `hdb_feature_table_*.csv` (same as training notebooks).

**Outputs:** `03_ml_layer_hybrid/artifacts/hybrid_xai/` — SHAP (per-cluster XGB TreeExplainer + global XGB for fallback), LIME, surrogate tree, Apriori rules, CBR index.

**SHAP trade-off:** Full hybrid = meta-stack over Ridge/XGB/LGB/RF. SHAP TreeExplainer applies to **XGBoost only** within each cluster (or global XGB when the route uses fallback). This parallels 06b’s “explain the dominant tree component.”


In [4]:
%pip install -q numpy pandas scikit-learn xgboost lightgbm shap lime mlxtend matplotlib joblib pyarrow

Note: you may need to restart the kernel to use updated packages.


In [5]:
import importlib
import json
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import lime
import lime.lime_tabular
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.neighbors import BallTree
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

warnings.filterwarnings("ignore")

import sys

# Repo root (cwd may be repo root, 04_xai_layer/, etc.)
_HERE = Path.cwd().resolve()
REPO_ROOT = _HERE if (_HERE / "hf_data").exists() else _HERE.parent
HYBRID_DIR = REPO_ROOT / "03_ml_layer_hybrid"
if HYBRID_DIR.exists() and str(HYBRID_DIR) not in sys.path:
    sys.path.insert(0, str(HYBRID_DIR))

import yc_hybrid_inference
importlib.reload(yc_hybrid_inference)
from yc_hybrid_inference import predict_price, load_bundle

HF_DATA_ROOT = REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs"


def _latest_feature_snapshot_date(root: Path) -> str:
    tables = sorted(root.glob("hdb_feature_table_*.csv"))
    if not tables:
        raise FileNotFoundError(f"No hdb_feature_table_*.csv under {root}")
    return tables[-1].stem.split("_")[-1]


_snap = _latest_feature_snapshot_date(HF_DATA_ROOT)
all_path = HF_DATA_ROOT / f"hdb_feature_table_{_snap}.csv"
if not all_path.exists():
    raise FileNotFoundError(f"Missing feature table: {all_path}")

HERE = REPO_ROOT
ART = HYBRID_DIR / "artifacts"
OUT_DIR = ART / "hybrid_xai"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = ART / "hybrid_cluster_bundle.joblib"

TARGET = "resale_price"
YEAR_COL = "transaction_year"

print("HF_DATA_ROOT:", HF_DATA_ROOT)
print("Feature table:", all_path.name)
print("Artifacts:", ART)
print("OUT_DIR:", OUT_DIR)

HF_DATA_ROOT: /Users/bhuvesh/Documents/PropertyLens/hf_data/02_feature_layer/training/outputs
Feature table: hdb_feature_table_20260406.csv
Artifacts: /Users/bhuvesh/Documents/PropertyLens/03_ml_layer_hybrid/artifacts
OUT_DIR: /Users/bhuvesh/Documents/PropertyLens/03_ml_layer_hybrid/artifacts/hybrid_xai


In [6]:
# Load full table (chronological order for splits); all_path from setup cell
df = pd.read_csv(all_path)
df = df.sort_values([YEAR_COL, "address_key"], kind="mergesort").reset_index(drop=True)

bundle = joblib.load(BUNDLE_PATH)
FEATURE_COLS = bundle["feature_columns"]
N_CLUSTERS = int(bundle["n_clusters"])

train_mask = df[YEAR_COL] < 2024
val_mask = df[YEAR_COL] == 2024
test_mask = df[YEAR_COL] >= 2025

X_all = df[FEATURE_COLS].fillna(0).astype(float)
y_all = df[TARGET].astype(float)

X_train = X_all.loc[train_mask].values
y_train = y_all.loc[train_mask].values
X_test = X_all.loc[test_mask].values
y_test = y_all.loc[test_mask].values
df_train = df.loc[train_mask].reset_index(drop=True)
df_test = df.loc[test_mask].reset_index(drop=True)

print("Train rows:", len(X_train), "| Test rows:", len(X_test), "| Features:", len(FEATURE_COLS))


def hybrid_predict(X):
    return predict_price(np.asarray(X, dtype=float), bundle)


# Sanity: hybrid test MAPE on a sample (predict_price loops rows)
_nq = min(3000, len(X_test))
_test_p = hybrid_predict(X_test[:_nq])
_mape = mean_absolute_percentage_error(y_test[:_nq], _test_p) * 100
print(f"Hybrid MAPE on first {_nq} test rows: {_mape:.2f}%")

Train rows: 204066 | Test rows: 29048 | Features: 77
Hybrid MAPE on first 3000 test rows: 4.45%


## 1 — SHAP: TreeExplainer per-cluster XGB + global XGB for fallback

In [7]:
cb = bundle["cluster_bundles"]
per_cluster_explainer = {}
fallback_ids = []

for k in range(N_CLUSTERS):
    c = cb[k] if k in cb else cb.get(str(k), {})
    if c.get("fallback"):
        fallback_ids.append(k)
        print(f"Cluster {k}: fallback → will use global XGB TreeExplainer")
    else:
        xgb_m = c["xgb"]
        per_cluster_explainer[k] = shap.TreeExplainer(xgb_m)
        print(f"Cluster {k}: TreeExplainer on XGB ✓")

# Global XGB (for fallback routes)
global_xgb = bundle["global_models"]["xgb"]
explainer_global_xgb = shap.TreeExplainer(global_xgb)
print("Global XGB TreeExplainer ✓")

shap_payload = {
    "per_cluster": per_cluster_explainer,
    "fallback_cluster_ids": fallback_ids,
    "global_xgb_explainer": explainer_global_xgb,
    "n_clusters": N_CLUSTERS,
}
joblib.dump(shap_payload, OUT_DIR / "shap_explainers.joblib")
print("Saved", OUT_DIR / "shap_explainers.joblib")

# Optional: global importance from a sample of test (using routed explainer)
rng = np.random.RandomState(42)
n_s = min(800, len(X_test))
idx = rng.choice(len(X_test), n_s, replace=False)
X_s = X_test[idx]
# Route each row to explainer
idx_feat = [FEATURE_COLS.index(c) for c in bundle["cluster_cols"]]
Xc = np.asarray(X_s[:, idx_feat], dtype=float)
sc = bundle["cluster_scaler"]
km = bundle["kmeans"]
labels = km.predict(sc.transform(Xc))
cl_bundles = bundle["cluster_bundles"]

abs_acc = np.zeros(len(FEATURE_COLS))
for i in range(len(X_s)):
    row = X_s[i : i + 1]
    k = int(labels[i])
    cbk = cl_bundles[k] if k in cl_bundles else cl_bundles[str(k)]
    if cbk.get("fallback"):
        exp = explainer_global_xgb
    else:
        exp = per_cluster_explainer[k]
    sv = exp(row)
    abs_acc += np.abs(sv.values[0])
abs_acc /= len(X_s)

shap_importance = dict(sorted(zip(FEATURE_COLS, abs_acc.tolist()), key=lambda x: x[1], reverse=True))
with open(OUT_DIR / "global_shap_cache.json", "w") as f:
    json.dump({k: float(v) for k, v in list(shap_importance.items())[:40]}, f, indent=2)

print("Top 5 |SHAP| (routed proxy):", list(shap_importance.items())[:5])

Cluster 0: TreeExplainer on XGB ✓
Cluster 1: TreeExplainer on XGB ✓
Cluster 2: fallback → will use global XGB TreeExplainer
Cluster 3: TreeExplainer on XGB ✓
Global XGB TreeExplainer ✓
Saved /Users/bhuvesh/Documents/PropertyLens/03_ml_layer_hybrid/artifacts/hybrid_xai/shap_explainers.joblib
Top 5 |SHAP| (routed proxy): [('transaction_year', 114497.84791015626), ('floor_area_sqm', 61811.82157897949), ('lease_remaining_years', 40300.43299129486), ('room_count', 30734.135173950195), ('dist_to_highway_m', 20779.08687932372)]


## 2 — LIME (hybrid predict on tabular samples)

In [8]:
LIME_SAMPLE_N = min(10_000, len(X_train))
rng = np.random.RandomState(42)
lime_idx = rng.choice(len(X_train), LIME_SAMPLE_N, replace=False)
lime_train = X_train[lime_idx]

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    lime_train,
    feature_names=FEATURE_COLS,
    mode="regression",
    discretize_continuous=True,
)

joblib.dump(
    {"training_data": lime_train, "feature_names": FEATURE_COLS},
    OUT_DIR / "lime_training_data.joblib",
)
print("Saved", OUT_DIR / "lime_training_data.joblib", f"shape={lime_train.shape}")

Saved /Users/bhuvesh/Documents/PropertyLens/03_ml_layer_hybrid/artifacts/hybrid_xai/lime_training_data.joblib shape=(10000, 77)


## 3 — Surrogate tree (fits hybrid train predictions)

In [9]:
# Subsample: hybrid_predict is O(n) row-wise — ≤25k train, ≤8k test
_rng = np.random.RandomState(42)
_n_tr = min(25_000, len(X_train))
_i_tr = _rng.choice(len(X_train), _n_tr, replace=False)
X_tr_sub = X_train[_i_tr]
hy_train_preds = hybrid_predict(X_tr_sub)
surrogate = DecisionTreeRegressor(max_depth=5, min_samples_leaf=100, random_state=42)
surrogate.fit(X_tr_sub, hy_train_preds)

_n_te = min(8_000, len(X_test))
_i_te = _rng.choice(len(X_test), _n_te, replace=False)
X_te_sub = X_test[_i_te]
hy_test_preds = hybrid_predict(X_te_sub)
surrogate_preds = surrogate.predict(X_te_sub)
surrogate_r2 = r2_score(hy_test_preds, surrogate_preds)
print(f"Surrogate fidelity R² (vs hybrid, subsampled): {surrogate_r2:.4f}")


def extract_surrogate_rules(tree, feature_names, max_rules=25):
    from sklearn.tree import _tree
    tree_ = tree.tree_
    rules = []

    def recurse(node, conds):
        if tree_.feature[node] == _tree.TREE_UNDEFINED:
            rules.append(
                {
                    "source": "surrogate",
                    "conditions": list(conds),
                    "then_price": round(float(tree_.value[node][0][0]), -3),
                    "samples": int(tree_.n_node_samples[node]),
                    "confidence": round(float(surrogate_r2), 3),
                }
            )
            return
        feat = feature_names[tree_.feature[node]]
        thresh = round(float(tree_.threshold[node]), 2)
        recurse(tree_.children_left[node], conds + [f"{feat} <= {thresh}"])
        recurse(tree_.children_right[node], conds + [f"{feat} > {thresh}"])

    recurse(0, [])
    rules.sort(key=lambda x: x["samples"], reverse=True)
    return rules[:max_rules]


surrogate_rules = extract_surrogate_rules(surrogate, FEATURE_COLS)
joblib.dump(surrogate, OUT_DIR / "surrogate_model.joblib")
print("Saved surrogate_model.joblib")

KeyboardInterrupt: 

## 4 — Apriori (training split — discretised bands; data-driven)

In [10]:
# Use columns available in the feature export
ap_df = df_train[
    ["floor_area_sqm", "level_mid", "lease_remaining_years", "dist_to_mrt_m", "room_count", "resale_price"]
].copy()

ap_df["area_band"] = pd.cut(
    ap_df["floor_area_sqm"],
    bins=[0, 70, 95, 999],
    labels=["area=small(<70sqm)", "area=medium(70-95sqm)", "area=large(>95sqm)"],
)
ap_df["storey_band"] = pd.cut(
    ap_df["level_mid"],
    bins=[0, 5, 12, 999],
    labels=["storey=low(1-5)", "storey=mid(6-12)", "storey=high(>12)"],
)
ap_df["lease_band"] = pd.cut(
    ap_df["lease_remaining_years"],
    bins=[0, 55, 70, 999],
    labels=["lease=short(<55yr)", "lease=medium(55-70yr)", "lease=long(>70yr)"],
)
ap_df["price_band"] = pd.cut(
    ap_df["resale_price"],
    bins=[0, 350000, 500000, 700000, 99999999],
    labels=["price=budget(<350k)", "price=mid(350-500k)", "price=expensive(500-700k)", "price=luxury(>700k)"],
)
ap_df["mrt_band"] = pd.cut(
    ap_df["dist_to_mrt_m"],
    bins=[0, 400, 800, 999999],
    labels=["mrt=walking(<400m)", "mrt=nearby(400-800m)", "mrt=far(>800m)"],
)
ap_df["room_band"] = ap_df["room_count"].round().astype(int).astype(str).map(lambda x: f"rooms={x}")

item_cols = ["area_band", "storey_band", "lease_band", "price_band", "mrt_band", "room_band"]
transactions = ap_df[item_cols].astype(str).values.tolist()

if len(transactions) > 150_000:
    rng_ap = np.random.RandomState(42)
    idx_ap = rng_ap.choice(len(transactions), 150_000, replace=False)
    transactions = [transactions[i] for i in idx_ap]

te = TransactionEncoder()
te_df = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)
te_df = te_df.loc[:, te_df.columns.str.lower() != "nan"]

frequent = apriori(te_df, min_support=0.05, use_colnames=True)
rules_df = association_rules(frequent, metric="confidence", min_threshold=0.6)
price_bands = {
    "price=budget(<350k)",
    "price=mid(350-500k)",
    "price=expensive(500-700k)",
    "price=luxury(>700k)",
}
rules_price = rules_df[rules_df["consequents"].apply(lambda x: bool(x & price_bands))].sort_values(
    "lift", ascending=False
)

apriori_rules_json = [
    {
        "source": "apriori",
        "if_conditions": list(r["antecedents"]),
        "then": list(r["consequents"]),
        "support": round(float(r["support"]), 4),
        "confidence": round(float(r["confidence"]), 3),
        "lift": round(float(r["lift"]), 3),
    }
    for _, r in rules_price.head(50).iterrows()
]
print(f"Apriori rules with price consequent: {len(rules_price)} (stored top {len(apriori_rules_json)})")

Apriori rules with price consequent: 36 (stored top 36)


## 5 — CBR (BallTree on interpretable numeric features)

In [11]:
CBR_FEATURES = [
    "floor_area_sqm",
    "level_mid",
    "lease_remaining_years",
    "dist_to_mrt_m",
    "dist_to_nearest_mall_m",
    "room_count",
    "market_activity_score",
]

cbr_df = df_train[CBR_FEATURES + ["resale_price", "address_key", YEAR_COL]].copy()
cbr_df = cbr_df.dropna(subset=CBR_FEATURES)

cbr_scaler = MinMaxScaler()
cbr_X = cbr_scaler.fit_transform(cbr_df[CBR_FEATURES].astype(float))
cbr_tree = BallTree(cbr_X, metric="euclidean")

joblib.dump(cbr_tree, OUT_DIR / "cbr_index.joblib")
joblib.dump(cbr_scaler, OUT_DIR / "cbr_scaler.joblib")
cbr_df.to_parquet(OUT_DIR / "cbr_training_data.parquet", index=False)
with open(OUT_DIR / "cbr_features.json", "w") as f:
    json.dump(CBR_FEATURES, f, indent=2)
print("Saved CBR artefacts")

Saved CBR artefacts


In [ ]:
# Combined rules + metadata
all_rules = {
    "apriori": apriori_rules_json,
    "surrogate": surrogate_rules,
    "metadata": {
        "apriori_count": len(apriori_rules_json),
        "surrogate_count": len(surrogate_rules),
        "surrogate_fidelity_r2": round(float(surrogate_r2), 4),
        "model": "yc_cluster_hybrid",
    },
}
with open(OUT_DIR / "rules.json", "w") as f:
    json.dump(all_rules, f, indent=2)
print("Saved rules.json")

meta = {
    "bundle_path": str(BUNDLE_PATH.relative_to(HERE)),
    "out_dir": str(OUT_DIR.relative_to(HERE)),
    "feature_count": len(FEATURE_COLS),
    "fallback_cluster_ids": fallback_ids,
}
with open(OUT_DIR / "hybrid_xai_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\n── Artefacts in hybrid_xai ──")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size/1024:,.0f} KB")

Saved rules.json

── Artefacts in hybrid_xai ──
  cbr_features.json: 0 KB
  cbr_index.joblib: 13,459 KB
  cbr_scaler.joblib: 1 KB
  cbr_training_data.parquet: 1,976 KB
  global_shap_cache.json: 2 KB
  hybrid_xai_meta.json: 0 KB
  lime_training_data.joblib: 6,017 KB
  rules.json: 18 KB
  shap_explainers.joblib: 245,753 KB
  surrogate_model.joblib: 5 KB
